# Cleaning Data

In [2]:
import pandas as pd

In [3]:
df=pd.read_csv("../data/raw_games.csv")

In [4]:
df.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,Season
0,22021,1610612744,GSW,Golden State Warriors,22100002,2021-10-19,GSW @ LAL,W,240,41,...,50,30,9,2,17,18,121,7,1,2021-22
1,22021,1610612747,LAL,Los Angeles Lakers,22100002,2021-10-19,LAL vs. GSW,L,240,45,...,45,21,7,4,18,25,114,-7,1,2021-22
2,22021,1610612749,MIL,Milwaukee Bucks,22100001,2021-10-19,MIL vs. BKN,W,240,48,...,54,25,8,9,8,19,127,23,1,2021-22
3,22021,1610612751,BKN,Brooklyn Nets,22100001,2021-10-19,BKN @ MIL,L,240,37,...,44,19,3,9,13,17,104,-23,1,2021-22
4,22021,1610612738,BOS,Boston Celtics,22100005,2021-10-20,BOS @ NYK,L,290,48,...,56,34,13,9,18,24,134,-4,1,2021-22


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12300 entries, 0 to 12299
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   SEASON_ID          12300 non-null  int64  
 1   TEAM_ID            12300 non-null  int64  
 2   TEAM_ABBREVIATION  12300 non-null  str    
 3   TEAM_NAME          12300 non-null  str    
 4   GAME_ID            12300 non-null  int64  
 5   GAME_DATE          12300 non-null  str    
 6   MATCHUP            12300 non-null  str    
 7   WL                 12300 non-null  str    
 8   MIN                12300 non-null  int64  
 9   FGM                12300 non-null  int64  
 10  FGA                12300 non-null  int64  
 11  FG_PCT             12300 non-null  float64
 12  FG3M               12300 non-null  int64  
 13  FG3A               12300 non-null  int64  
 14  FG3_PCT            12300 non-null  float64
 15  FTM                12300 non-null  int64  
 16  FTA                12300 non-null

In [6]:
df=df.drop(columns=['VIDEO_AVAILABLE', 'Season', 'SEASON_ID', 'TEAM_ID', 'TEAM_NAME']) # Dropping non useful column
df=df.drop(columns=['PLUS_MINUS']) # this gives away what we want to predict 
df

,TEAM_ABBREVIATION,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS
0,GSW,22100002,2021-10-19,GSW @ LAL,W,240,41,93,0.441,14,...,0.833,9,41,50,30,9,2,17,18,121
1,LAL,22100002,2021-10-19,LAL vs. GSW,L,240,45,95,0.474,15,...,0.474,5,40,45,21,7,4,18,25,114
2,MIL,22100001,2021-10-19,MIL vs. BKN,W,240,48,105,0.457,17,...,0.778,13,41,54,25,8,9,8,19,127
3,BKN,22100001,2021-10-19,BKN @ MIL,L,240,37,84,0.440,17,...,0.565,5,39,44,19,3,9,13,17,104
4,BOS,22100005,2021-10-20,BOS @ NYK,L,290,48,117,0.410,21,...,0.739,15,41,56,34,13,9,18,24,134
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12295,MIL,22501191,2026-04-12,MIL @ PHI,L,240,39,90,0.433,13,...,0.682,12,33,45,26,8,2,19,15,106
12296,POR,22501200,2026-04-12,POR vs. SAC,W,240,47,101,0.465,16,...,0.800,18,28,46,28,9,3,12,21,122
12297,PHX,22501196,2026-04-12,PHX @ OKC,W,240,56,101,0.554,20,...,0.500,13,41,54,28,5,8,13,9,135
12298,CLE,22501187,2026-04-12,CLE vs. WAS,W,240,48,89,0.539,18,...,0.762,17,36,53,31,8,4,20,18,130


In [7]:
df["WL"]=df['WL'].map({'W':1,'L':0})# W=1 L=0
# Identifying Home and Away team
home_df=df[df['MATCHUP'].str.contains('vs.')].copy()
away_df=df[df['MATCHUP'].str.contains('@')].copy()
# identify columns having same values in both entries
merge_cols=['GAME_ID']

# Add HOME_ prefix, but rename the merge columns back to normal
home_df = home_df.add_prefix('HOME_')
home_df = home_df.rename(columns={ 
    'HOME_GAME_ID': 'GAME_ID'
})
# Add AWAY_ prefix, but rename the merge columns back to normal
away_df = away_df.add_prefix('AWAY_')
away_df = away_df.rename(columns={ 
    'AWAY_GAME_ID': 'GAME_ID'
})
# merging
games_df=pd.merge(home_df, away_df, on=merge_cols)

print(f"Original shape: {df.shape}, New shape: {games_df.shape}")
games_df.info()

Original shape: (12300, 24), New shape: (6140, 47)
<class 'pandas.DataFrame'>
RangeIndex: 6140 entries, 0 to 6139
Data columns (total 47 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   HOME_TEAM_ABBREVIATION  6140 non-null   str    
 1   GAME_ID                 6140 non-null   int64  
 2   HOME_GAME_DATE          6140 non-null   str    
 3   HOME_MATCHUP            6140 non-null   str    
 4   HOME_WL                 6140 non-null   int64  
 5   HOME_MIN                6140 non-null   int64  
 6   HOME_FGM                6140 non-null   int64  
 7   HOME_FGA                6140 non-null   int64  
 8   HOME_FG_PCT             6140 non-null   float64
 9   HOME_FG3M               6140 non-null   int64  
 10  HOME_FG3A               6140 non-null   int64  
 11  HOME_FG3_PCT            6140 non-null   float64
 12  HOME_FTM                6140 non-null   int64  
 13  HOME_FTA                6140 non-null   int64  
 14  

In [8]:
# Exhibition Games
# Find which game IDs are not in both
home_games = set(home_df['GAME_ID'])
away_games = set(away_df['GAME_ID'])

missing_in_away = home_games - away_games
missing_in_home = away_games - home_games

print(f"Games with no away row: {missing_in_away}")
print(f"Games with no home row: {missing_in_home}")

for gid in missing_in_away.union(missing_in_home):
    print(df[df['GAME_ID'] == gid][['GAME_ID', 'MATCHUP', 'TEAM_ABBREVIATION']])

Games with no away row: set()
Games with no home row: {22500578, 22401229, 22400621, 22401230, 22501229, 22501230, 22400147, 22500147, 22400633, 22500602}
        GAME_ID    MATCHUP TEAM_ABBREVIATION
11050  22500578  MEM @ ORL               MEM
11052  22500578  ORL @ MEM               ORL
       GAME_ID    MATCHUP TEAM_ABBREVIATION
8128  22401229  ATL @ MIL               ATL
8130  22401229  MIL @ ATL               MIL
       GAME_ID    MATCHUP TEAM_ABBREVIATION
8681  22400621  IND @ SAS               IND
8684  22400621  SAS @ IND               SAS
       GAME_ID    MATCHUP TEAM_ABBREVIATION
8129  22401230  HOU @ OKC               HOU
8131  22401230  OKC @ HOU               OKC
        GAME_ID    MATCHUP TEAM_ABBREVIATION
10584  22501229  ORL @ NYK               ORL
10585  22501229  NYK @ ORL               NYK
        GAME_ID    MATCHUP TEAM_ABBREVIATION
10586  22501230  OKC @ SAS               OKC
10587  22501230  SAS @ OKC               SAS
       GAME_ID    MATCHUP TEAM_ABBREVIATION


In [9]:
# games_df = games_df.drop(columns=['AWAY_WL'])
games_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6140 entries, 0 to 6139
Data columns (total 47 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   HOME_TEAM_ABBREVIATION  6140 non-null   str    
 1   GAME_ID                 6140 non-null   int64  
 2   HOME_GAME_DATE          6140 non-null   str    
 3   HOME_MATCHUP            6140 non-null   str    
 4   HOME_WL                 6140 non-null   int64  
 5   HOME_MIN                6140 non-null   int64  
 6   HOME_FGM                6140 non-null   int64  
 7   HOME_FGA                6140 non-null   int64  
 8   HOME_FG_PCT             6140 non-null   float64
 9   HOME_FG3M               6140 non-null   int64  
 10  HOME_FG3A               6140 non-null   int64  
 11  HOME_FG3_PCT            6140 non-null   float64
 12  HOME_FTM                6140 non-null   int64  
 13  HOME_FTA                6140 non-null   int64  
 14  HOME_FT_PCT             6140 non-null   float64
 15

In [10]:
# Sort the dataframe from oldest games to newest games
games_df = games_df.sort_values('HOME_GAME_DATE')

# Reset the index so your row numbers start cleanly from 0 again
games_df = games_df.reset_index(drop=True)
games_df

,HOME_TEAM_ABBREVIATION,GAME_ID,HOME_GAME_DATE,HOME_MATCHUP,HOME_WL,HOME_MIN,HOME_FGM,HOME_FGA,HOME_FG_PCT,HOME_FG3M,...,AWAY_FT_PCT,AWAY_OREB,AWAY_DREB,AWAY_REB,AWAY_AST,AWAY_STL,AWAY_BLK,AWAY_TOV,AWAY_PF,AWAY_PTS
0,LAL,22100002,2021-10-19,LAL vs. GSW,0,240,45,95,0.474,15,...,0.833,9,41,50,30,9,2,17,18,121
1,MIL,22100001,2021-10-19,MIL vs. BKN,1,240,48,105,0.457,17,...,0.565,5,39,44,19,3,9,13,17,104
2,CHA,22100003,2021-10-20,CHA vs. IND,1,240,46,107,0.430,13,...,0.875,8,43,51,29,2,10,17,24,122
3,DET,22100004,2021-10-20,DET vs. CHI,0,240,36,90,0.400,6,...,0.867,9,39,48,18,8,5,17,19,94
4,MEM,22100007,2021-10-20,MEM vs. CLE,1,240,53,100,0.530,14,...,0.722,7,29,36,38,6,5,11,17,121
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6135,NYK,22501190,2026-04-12,NYK vs. CHA,0,240,39,90,0.433,13,...,0.765,12,34,46,27,7,7,18,13,110
6136,BOS,22501186,2026-04-12,BOS vs. ORL,1,240,36,87,0.414,19,...,0.800,14,36,50,22,6,5,19,19,108
6137,CLE,22501187,2026-04-12,CLE vs. WAS,1,240,48,89,0.539,18,...,0.789,7,23,30,29,10,2,13,21,117
6138,MIN,22501195,2026-04-12,MIN vs. NOP,1,240,45,89,0.506,10,...,0.816,20,32,52,20,9,6,9,28,126


In [11]:
# Save the intermediate merged dataset without the pandas index column
games_df.to_csv("../data/merged_games_intermediate.csv", index=False)